# Neighborhood calculation for ULMS G4X data
- This is for the revision of the ULMS paper

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq

sc.settings.n_jobs = -1  # Use all available cores
plt.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.ioff()
sc.settings.autoshow = False

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpasq

# version control
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("squidpy:", sq.__version__)

pandas: 2.3.2
numpy: 2.2.6
scanpy: 1.11.4
squidpy: 1.8.1


In [2]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
print(PARENT_DIR)

OUTPUT_MASTER_DIR = jpasq.create_output_dir(PARENT_DIR, 'neighborhood', change_dir=True)
DATA_DIR = PARENT_DIR.parent.parent / 'G4X' / 'G4X_raw'
print(DATA_DIR)
ANNDATA_DIR = PARENT_DIR / 'objects'
print(ANNDATA_DIR)

/oak/stanford/groups/longaker/ULMS/revision/G4X
Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/neighborhood
Working directory and scanpy figure output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/neighborhood
/oak/stanford/groups/longaker/ULMS/G4X/G4X_raw
/oak/stanford/groups/longaker/ULMS/revision/G4X/objects


# Set up

In [4]:
all_cells = sc.read_h5ad(ANNDATA_DIR / 'g4x_all_raw.h5ad')
all_cells

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'Sample', 'ct'
    uns: 'ct_colors'
    obsm: 'X_spatial'

In [5]:
# order sections properly. This is how they will print on the page.
sections = [
    'B01', 'C01', 'D01', 'E01', 
    'F01', 'G01', 'H01', 'A02', 
    'B02', 'C02', 'E02', 'F02', 
    'G02', 'H02', 'A03', 'B03', 
    'C03', 'D03', 'E03', 'F03', 
    'G03', 'H03', 'A04', 'B04', 
    'C04', 'D04',
]

In [ ]:
# # Color palette for celltypes
# # Make sure the celltype colors line up
# ct_categories = all_cells.obs['cell_type'].cat.categories
# ct_colors = all_cells.uns['cell_type_colors']
# ct_palette = dict(zip(ct_categories, ct_colors))
# ct_palette

In [ ]:
all_cell_types = all_cells.obs['cell_type'].cat.categories.tolist()
all_cell_types

# Neighborhood analysis for all tumor subtypes and all cell types together

In [ ]:
# This cell is going to do neighborhood enrichment analysis section by section and store the result
scores = {} # stores neighorhood enrichment Z-scores

# https://stats.stackexchange.com/questions/312605/can-i-treat-the-mean-of-a-set-of-z-scores-as-a-z-score
# nan issue with computing the dendrogram in the plot command: https://github.com/scverse/squidpy/issues/557
for section in sections:
    print(section)
    # subset the annotated anndata
    adata = all_cells[all_cells.obs['Section'] == section].copy()
    # set 'spatial' as the spatial key for squidpy
    adata.obsm['spatial'] = adata.obsm.pop('X_spatial')

    # filter out low-frequency subtypes. This avoids issues with the SD being 0 and creating division by 0 leading to nan or infinity.
    type_counts = adata.obs['cell_type'].value_counts()
    top_types = type_counts[type_counts > 100].index.tolist()
    adata = adata[adata.obs['cell_type'].isin(top_types)]
    
    # compute the spatial neighborhood connectivity graph
    sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True, spatial_key='spatial',)
    # compute the neighborhood enrichment scores for each section
    sq.gr.nhood_enrichment(adata, cluster_key='cell_type', connectivity_key='spatial', seed=1234,)
    # plot the neighborhood enrichment z-scores for each section
    sq.pl.nhood_enrichment(adata, cluster_key='cell_type', method='mean', save=f'{section}_finecelltype_nhood.png', dpi=300)
    plt.close()
    
    # store the subtypes - list
    section_types = top_types
    # store the Z-scores - array
    section_zscores = adata.uns['cell_type_nhood_enrichment']['zscore']

    # create a square matrix with subtypes in that section as the row names and column names
    df = pd.DataFrame(section_zscores, index=section_types, columns=section_types)
    # parallelize all the z_score matrices so that they contain the same rows and columns.
    # if the subtypes were not in that section, the z_score will be set to 0.0
    for t in all_cell_types:
        if t not in section_types: # add missing subtypes to the data frame with zeros
            df.loc[t] = 0.0
            df[t] = 0.0
    df = df.reindex(index=all_cell_types, columns=all_cell_types) # reorder rows and columns to match the order of the cell types
    scores[section] = df # store the dataframe for that section in the dictionary

In [ ]:
# https://stackoverflow.com/questions/25057835/get-the-mean-across-multiple-pandas-dataframes?newreg=b8540c3f08fc4c28a81b20ba3d839ab6
df = pd.concat(list(scores.values()))
df = df.groupby(level=0).mean()
df = df.reindex(index=all_cell_types, columns=all_cell_types) # reorder rows and columns, since the previous two lines will put them out of order
all_cells.uns['cell_type_nhood_enrichment'] = {}
all_cells.uns['cell_type_nhood_enrichment']['zscore'] = df.to_numpy()

In [ ]:
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', method='median', save='allcells_fine_celltype_nhood_median.png', dpi=300)
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', method='median', save='allcells_fine_celltype_nhood_median.pdf', dpi=300)
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', method='average', save='allcells_fine_celltype_nhood_avg.png', dpi=300)
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', method='average', save='allcells_fine_celltype_nhood_avg.pdf', dpi=300)
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', save='allcells_fine_celltype_nhood.png', dpi=300) # no dendrogram
sq.pl.nhood_enrichment(all_cells, cluster_key='cell_type', save='allcells_fine_celltype_nhood.pdf', dpi=300) # no dendrogram

# Example: B01 cell type neighborhood analysis

In [ ]:
# # subset the annotated anndata
# section = 'B01'
# adata = all_cells[all_cells.obs['Section'] == section].copy()
# adata.obsm['spatial'] = adata.obsm.pop('X_spatial')
# adata

In [ ]:
# # filter out low-frequency cell types
# celltype_counts = adata.obs['celltype'].value_counts()
# top_celltypes = celltype_counts[celltype_counts > 100].index.tolist()
# adata = adata[adata.obs['celltype'].isin(top_celltypes)]
# sq.pl.spatial_scatter(adata, library_id='spatial', shape=None, color='celltype', save=f'{section}_sp_scatter.png', dpi=300, frameon=False)

In [ ]:
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True, spatial_key='spatial',)
# sq.gr.nhood_enrichment(adata, cluster_key='celltype', connectivity_key='spatial')

In [ ]:
# sq.pl.nhood_enrichment(adata, cluster_key='celltype', method='average', save=f'{section}_nhood.png', dpi=300) # with the dendrogram

In [ ]:
# arr = adata.uns['celltype_nhood_enrichment']['count']
# min_val = arr.min()
# max_val = arr.max()
# scaled_arr = 2 * (arr - min_val) / (max_val - min_val) - 1
# print(scaled_arr)

In [ ]:
# adata.uns['celltype_nhood_enrichment']['zscore'] = scaled_arr

In [ ]:
# sq.pl.nhood_enrichment(adata, cluster_key='celltype', method='average') # with the dendrogram

# Example: E02 tumor subtype neighborhood analysis

In [ ]:
# # subset the annotated anndata
# section = 'E02'
# adata = all_cells[all_cells.obs['Section'] == section].copy()
# adata.obsm['spatial'] = adata.obsm.pop('X_spatial')
# adata

In [ ]:
# # filter out low-frequency subtypes
# subtype_counts = adata.obs['tumor_subtype'].value_counts()
# top_subtypes = subtype_counts[subtype_counts > 100].index.tolist()
# adata = adata[adata.obs['tumor_subtype'].isin(top_subtypes)]
# sq.pl.spatial_scatter(adata, library_id='spatial', shape=None, color='tumor_subtype', save=f'{section}_sp_scatter.png', dpi=300, frameon=False)

In [ ]:
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True, spatial_key='spatial',)
# sq.gr.nhood_enrichment(adata, cluster_key='tumor_subtype', connectivity_key='spatial')
# sq.pl.nhood_enrichment(adata, cluster_key='tumor_subtype', method='average', save=f'{section}_nhood.png', dpi=300) # with the dendrogram

In [ ]:
# matrix = adata.uns['tumor_subtype_nhood_enrichment']['zscore']
# np.mean(matrix)

# Neighborhood analysis for all cell types in all sections

In [ ]:
# ct_cats = all_cells.obs['cell_type'].cat.categories.tolist()
# ct_cats

In [ ]:
# scores = {} # stores neighorhood enrichment Z-scores

# # https://stats.stackexchange.com/questions/312605/can-i-treat-the-mean-of-a-set-of-z-scores-as-a-z-score
# # nan issue with computing the dendrogram in the plot command: https://github.com/scverse/squidpy/issues/557
# for section in sections:
#     print(section)
#     # subset the annotated anndata
#     adata = all_cells[all_cells.obs['Section'] == section].copy()
#     # set 'spatial' as the spatial key for squidpy
#     adata.obsm['spatial'] = adata.obsm.pop('X_spatial')

#     # filter out low-frequency cell types. This avoids issues with the SD being 0 and creating division by 0 leading to nan or infinity.
#     celltype_counts = adata.obs['celltype'].value_counts()
#     top_celltypes = celltype_counts[celltype_counts > 100].index.tolist()
#     adata = adata[adata.obs['celltype'].isin(top_celltypes)]
    
#     # compute the spatial neighborhood connectivity graph
#     sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True, spatial_key='spatial',)
#     # compute the neighborhood enrichment scores for each section
#     sq.gr.nhood_enrichment(adata, cluster_key='celltype', connectivity_key='spatial', seed=1234,)
#     # plot the neighborhood enrichment z-scores for each section
#     sq.pl.nhood_enrichment(adata, cluster_key='celltype', method='average', save=f'{section}_celltype_nhood.png', dpi=300)
#     plt.close()
    
#     # store the cell types - list
#     section_celltypes = adata.obs['celltype'].cat.categories.tolist()
#     # store the Z-scores - array
#     section_zscores = adata.uns['celltype_nhood_enrichment']['zscore']

#     # create a square matrix with cell types in that section as the row names and column names
#     df = pd.DataFrame(section_zscores, index=section_celltypes, columns=section_celltypes)
#     # parallelize all the z_score matrices so that they contain the same rows and columns.
#     # if the cell types were not in that section, the z_score will be set to 0.0
#     for ct in ct_cats:
#         if ct not in section_celltypes: # add missing cell types to the data frame with zeros
#             df.loc[ct] = 0.0
#             df[ct] = 0.0
#     df = df[ct_cats] # reorder columns
#     df = df.reindex(ct_cats) # reorder rows
#     scores[section] = df # store the dataframe in the dictionary

In [ ]:
# # https://stackoverflow.com/questions/25057835/get-the-mean-across-multiple-pandas-dataframes?newreg=b8540c3f08fc4c28a81b20ba3d839ab6
# df = pd.concat(list(scores.values()))
# df = df.groupby(level=0).mean()
# all_cells.uns['celltype_nhood_enrichment'] = {}
# all_cells.uns['celltype_nhood_enrichment']['zscore'] = df.to_numpy()

In [ ]:
# sq.pl.nhood_enrichment(all_cells, cluster_key='celltype', method='average', save='allcells_celltype_nhood.png', dpi=300) # with the dendrogram

# Neighborhood analysis for all tumor subtypes in all sections

In [ ]:
# st_cats = all_cells.obs['tumor_subtype'].cat.categories.tolist()
# st_cats

In [ ]:
# scores = {} # stores neighorhood enrichment Z-scores

# # https://stats.stackexchange.com/questions/312605/can-i-treat-the-mean-of-a-set-of-z-scores-as-a-z-score
# # nan issue with computing the dendrogram in the plot command: https://github.com/scverse/squidpy/issues/557
# for section in sections:
#     print(section)
#     # subset the annotated anndata
#     adata = all_cells[all_cells.obs['Section'] == section].copy()
#     # set 'spatial' as the spatial key for squidpy
#     adata.obsm['spatial'] = adata.obsm.pop('X_spatial')

#     # filter out low-frequency subtypes. This avoids issues with the SD being 0 and creating division by 0 leading to nan or infinity.
#     subtype_counts = adata.obs['tumor_subtype'].value_counts()
#     top_subtypes = subtype_counts[subtype_counts > 100].index.tolist()
#     adata = adata[adata.obs['tumor_subtype'].isin(top_subtypes)]
    
#     # compute the spatial neighborhood connectivity graph
#     sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True, spatial_key='spatial',)
#     # compute the neighborhood enrichment scores for each section
#     sq.gr.nhood_enrichment(adata, cluster_key='tumor_subtype', connectivity_key='spatial', seed=1234,)
#     # plot the neighborhood enrichment z-scores for each section
#     sq.pl.nhood_enrichment(adata, cluster_key='tumor_subtype', method='average', save=f'{section}_subtype_nhood.png', dpi=300)
#     plt.close()
    
#     # store the subtypes - list
#     section_subtypes = adata.obs['tumor_subtype'].cat.categories.tolist()
#     # store the Z-scores - array
#     section_zscores = adata.uns['tumor_subtype_nhood_enrichment']['zscore']

#     # create a square matrix with subtypes in that section as the row names and column names
#     df = pd.DataFrame(section_zscores, index=section_subtypes, columns=section_subtypes)
#     # parallelize all the z_score matrices so that they contain the same rows and columns.
#     # if the subtypes were not in that section, the z_score will be set to 0.0
#     for st in st_cats:
#         if st not in section_subtypes: # add missing subtypes to the data frame with zeros
#             df.loc[st] = 0.0
#             df[st] = 0.0
#     df = df[st_cats] # reorder columns
#     df = df.reindex(st_cats) # reorder rows
#     scores[section] = df # store the dataframe in the dictionary

In [ ]:
# # https://stackoverflow.com/questions/25057835/get-the-mean-across-multiple-pandas-dataframes?newreg=b8540c3f08fc4c28a81b20ba3d839ab6
# df = pd.concat(list(scores.values()))
# df = df.groupby(level=0).mean()
# all_cells.uns['tumor_subtype_nhood_enrichment'] = {}
# all_cells.uns['tumor_subtype_nhood_enrichment']['zscore'] = df.to_numpy()

In [ ]:
# sq.pl.nhood_enrichment(all_cells, cluster_key='tumor_subtype', method='average', save='allcells_subtype_nhood.png', dpi=300) # with the dendrogram